# Modules

In [ ]:
from UnitTest import Timer, Duur, meet_duur,toon_duur, convert_wkt_string_to_gps, batch_transform_geometry_to_gps_string, convert_coords_to_gps

from scipy.spatial.distance import cdist
from IPython.display import display
import geopandas as gpd
from pyproj import Transformer
from shapely import wkt
from shapely.geometry.base import BaseGeometry
import re
from scipy.spatial import cKDTree
import zipfile
import urllib.request
from pathlib import Path
from shapely.geometry import LineString
from shapely.ops import transform as shapely_transform
import pandas as pd
import matplotlib as plt
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)       # Geen omvouwing naar volgende regel
pd.set_option('display.float_format', '{:.6f}'.format)  # 6 decimalen voor floats
pd.set_option('display.max_seq_items', None)
pd.set_option('display.precision', 10) 
pd.set_option('display.show_dimensions', True)
import numpy as np
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

# 1 Wegvakken inladen & filteren
Onderstaande cel kan je  beter maar één keer aan het begin draaien omdat deze zo slaam is omdat deze alle straten in heel Nederland inleest.

Nadat je onderstaande cel al één keer eerder hebt gedraait heb je al de gefilterde data die je nodig hebt. daarom kan je vnaf daarna beter eerst de bovenste cel draaien en dan vanaf na deze onderstaande cel alles in éen keer runnen.

In [ ]:
# wegvakken_ = gpd.read_file(
#     "Input/Beschrijvende_Plaatsaanduiding_systematiek/nwb-wegen-08_01_2026.gpkg",
#     layer="wegvakken")
# print(wegvakken_.columns)

# wegvakken = wegvakken_.copy()
# wegvakken = wegvakken[wegvakken['wegnr_hmp'].isin(['A12', 'A4', 'A20', 'N11',
#                                                    'N14', 'A13', 'A44', 'A16', 'N44',
#                                                    'A27', 'A2'])]
# wegvakken = wegvakken.rename(columns={'geometry': 'wegvak_geometry'})
# wegvakken = wegvakken[['wvk_id', 'rijrichtng', 'wegnummer', 'wegnr_hmp', 'wegnr_aw',
#                     'beginkm', 'eindkm', 'wegbehnaam', 'distrnaam', 'wegvak_geometry']]
# wegvakken.to_pickle("Output/wegvakken.pkl")
# wegvakken.sample(1).T

Index(['objectid', 'wvk_id', 'wvk_begdat', 'jte_id_beg', 'jte_id_end',
       'wegbehsrt', 'wegnummer', 'wegdeelltr', 'hecto_lttr', 'bst_code',
       'rpe_code', 'admrichtng', 'rijrichtng', 'stt_naam', 'stt_bron',
       'wpsnaam', 'gme_id', 'gme_naam', 'hnrstrlnks', 'hnrstrrhts',
       'e_hnr_lnks', 'e_hnr_rhts', 'l_hnr_lnks', 'l_hnr_rhts', 'begafstand',
       'endafstand', 'beginkm', 'eindkm', 'pos_tv_wol', 'wegbehcode',
       'wegbehnaam', 'distrcode', 'distrnaam', 'dienstcode', 'dienstnaam',
       'wegtype', 'wgtype_oms', 'routeltr', 'routenr', 'routeltr2', 'routenr2',
       'routeltr3', 'routenr3', 'routeltr4', 'routenr4', 'wegnr_aw',
       'wegnr_hmp', 'geobron_id', 'geobron_nm', 'bronjaar', 'openlr',
       'bag_orl', 'frc', 'fow', 'alt_naam', 'alt_nr', 'rel_hoogte',
       'st_lengthshape', 'geometry'],
      dtype='str')

In [ ]:
wegvakken = pd.read_pickle("Output/wegvakken.pkl")
print(f'Wegvakken geladen : {wegvakken.shape[0]:,} rijen, \
{wegvakken.shape[1]} kolommen')
wegvakken.sample(1).T

# 2.3 Hectopunten inladen, schoonmaken & koppelen aan wegvakken

In [ ]:
hectopunten_ = gpd.read_file("Input/Beschrijvende_Plaatsaanduiding_systematiek/nwb-wegen-08_01_2026.gpkg", layer='hectopunten')
hectopunten = hectopunten_.copy()

In [ ]:
print(f'Hectopunten geladen: {hectopunten.shape[0]:,} rijen, {hectopunten.shape[1]} kolommen')
display(hectopunten.sample(1).T)

In [ ]:
hectopunten = hectopunten.rename(columns={
    'geometry': 'hectopunt_geometry',
})
hectopunten = hectopunten.sort_values(by='wvk_id', ascending=True).reset_index()
hectopunten['meter'] = hectopunten['hectomtrng'] * 1000
hectopunten = hectopunten.astype({'wvk_id': 'int',
                                  'hectomtrng': 'int',
                                  'hecto_lttr': 'string',
                                  'hecto_lttr': 'string'})
hectopunten = hectopunten[['hectomtrng', 'afstand', 'wvk_id', 'hecto_lttr', 'hectopunt_geometry']]

In [ ]:
hectopunten.head(1).T

In [ ]:
hectopunten_voor = len(hectopunten)
wegvakken_voor = len(wegvakken)
df = hectopunten.merge(
    wegvakken, 
    on='wvk_id',
    how='inner'
)

# Duplicaten berekenen
aantal_duplicaten = df['wvk_id'].duplicated().sum()
heeft_duplicaten = "Ja" if aantal_duplicaten > 0 else "Nee"

print(f'Hectopunten voor merge : {hectopunten_voor:,}')
print(f'Wegvakken  voor merge  : {wegvakken_voor:,}')
print(f'Hectopunten na merge   : {df.shape[0]:,}')
print(f'Verlies hectopunten    : {hectopunten_voor - df.shape[0]:,} \
({(hectopunten_voor-df.shape[0])/hectopunten_voor*100:.1f}%)')
print(f'Kolommen na merge ({len(df.columns)}): {df.columns.tolist()}')
print(f'Duplicaten in wvk_id   : {heeft_duplicaten} ({aantal_duplicaten:,} stuks)')
if aantal_duplicaten > 0:
    print('\nVoorbeelden van duplicaten:')
    display(df[df['wvk_id'].duplicated(keep=False)].sort_values('wvk_id').head(4).T)

In [ ]:
df['Zijde'] = np.where(df['beginkm'] < df['eindkm'], 'Rechts', 'Links')
df['oplopend'] = np.where(df['beginkm'] < df['eindkm'], 'R', 'L')
df['snelwegnummer'] = df['wegnr_hmp'].str.replace(r'\D', '', regex=True).astype(int)
df['|'] = ''
df['info'] = ''
df.sample(1).T

# Dataset verrijken

In [ ]:
import numpy as np
from pyproj import Transformer

_TRANSFORMER = Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True)
_OFFSET_X, _OFFSET_Y = 89.35, 128.84
_NWB_UUID = "f2437a92-ddd3-4777-a1bc-fdf4b4a7fcb8"
_LAYERS = f"{_NWB_UUID};hectopunten;_;1,{_NWB_UUID};wegvakken;_;1"
_ZOOM = 12.5

geoms = df['hectopunt_geometry'].to_numpy()
n = len(geoms)
xs = np.full(n, np.nan)
ys = np.full(n, np.nan)
for i, g in enumerate(geoms):
    if g is None:
        continue
    gt = g.geom_type
    if gt == 'Point':
        xs[i] = g.x
        ys[i] = g.y
    elif gt == 'MultiPoint':
        p = g.geoms[0]
        xs[i] = p.x
        ys[i] = p.y

valid = ~np.isnan(xs)
lons = np.full(n, np.nan)
lats = np.full(n, np.nan)
if valid.any():
    lons[valid], lats[valid] = _TRANSFORMER.transform(xs[valid], ys[valid])

rd_coords        = [None] * n
gps_coords       = [None] * n
google_links     = [None] * n
streetsmart_links = [None] * n
pdok_links       = [None] * n

for i in range(n):
    if not valid[i]:
        continue
    x = round(xs[i], 2)
    y = round(ys[i], 2)
    lat = lats[i]
    lon = lons[i]
    gps = f"{lat:.6f}, {lon:.6f}"
    rd_coords[i]    = f"{x:.2f}, {y:.2f}"
    gps_coords[i]   = gps
    google_links[i] = f"https://www.google.com/maps/search/?api=1&query={gps}"
    minX = round(x - _OFFSET_X, 2)
    minY = round(y - _OFFSET_Y, 2)
    maxX = round(x + _OFFSET_X, 2)
    maxY = round(y + _OFFSET_Y, 2)
    streetsmart_links[i] = (
        f"https://streetsmart.cyclomedia.com/streetsmart/"
        f"?mq={minX};{minY};{maxX};{maxY}&msrs=EPSG:28992"
    )
    pdok_links[i] = (
        f"https://app.pdok.nl/viewer/#x={x:.2f}&y={y:.2f}&z={_ZOOM}"
        f"&background=BRT-A%20standaard&layers={_LAYERS}"
    )

df['gps_coordinaten']         = gps_coords
df['google_maps_link']        = google_links
df['hectopunt_rd_coordinaten'] = rd_coords
df['streetsmart_link']        = streetsmart_links
df['pdok_viewer_link']        = pdok_links

df.to_pickle("Output/Data_Analyse-Dataset.pkl")

# Inweva

In [ ]:
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'hectopunten geladen : {df.shape[0]:,} rijen, \
{df.shape[1]} kolommen')

In [ ]:
JAAR = 2025
INWEVA_DIR = Path("Input/Inweva")
INWEVA_DIR.mkdir(parents=True, exist_ok=True)
INWEVA_ZIP = INWEVA_DIR / f'INWEVA_{JAAR}.zip'
INWEVA_EXTRACTED = INWEVA_DIR / f'INWEVA_{JAAR}'
URL = f'https://downloads.rijkswaterstaatdata.nl/inweva/INWEVA_{JAAR}.zip'


def download_inweva():
    if not INWEVA_ZIP.exists():
        with Duur('Download INWEVA'):
            urllib.request.urlretrieve(URL, INWEVA_ZIP)
    if not INWEVA_EXTRACTED.exists():
        with zipfile.ZipFile(INWEVA_ZIP) as z:
            z.extractall(INWEVA_DIR)


def lees_netwerk():
    shp_pad = INWEVA_EXTRACTED / f'INWEVA_{JAAR}_netwerk' / f'INWEVA{JAAR}.shp'
    with Duur('Inlezen INWEVA shapefile'):
        netwerk = gpd.read_file(shp_pad)
    netwerk['VBN_ID'] = netwerk['VBN_ID'].astype(str)
    return netwerk


def lees_werkdag():
    return pd.read_csv(INWEVA_EXTRACTED / f'inweva_werkdag_{JAAR}DEC.csv',
                       sep=';', dtype={'VBN_ID': str})


def bouw_inweva_per_wvk(netwerk, werkdag):
    inweva = netwerk.merge(
        werkdag[['VBN_ID', 'AL_E_WR', 'L1_E_WR', 'L2_E_WR', 'L3_E_WR']],
        on='VBN_ID', how='left'
    )
    inweva_wvk = (
        inweva
        .assign(WVK_ID=inweva['WVK_IDS'].fillna('').str.split(','))
        .explode('WVK_ID')
    )
    inweva_wvk['WVK_ID'] = inweva_wvk['WVK_ID'].str.strip()
    inweva_wvk = inweva_wvk[inweva_wvk['WVK_ID'] != '']
    inweva_wvk['wvk_id'] = pd.to_numeric(inweva_wvk['WVK_ID'], errors='coerce').astype('Int64')

    per_wvk = (
        inweva_wvk[['wvk_id', 'L1_E_WR', 'L2_E_WR', 'L3_E_WR', 'AL_E_WR', 'VBNOMSTXT']]
        .drop_duplicates(subset='wvk_id', keep='first')
        .reset_index(drop=True)
        .rename(columns={
            'L1_E_WR': 'klein_voertuig',
            'L2_E_WR': 'middel_voertuig',
            'L3_E_WR': 'lang_voertuig',
            'AL_E_WR': 'totaal_voertuig',
            'VBNOMSTXT': '_inweva_omschrijving',
        })
    )
    voertuig_cols = ['klein_voertuig', 'middel_voertuig', 'lang_voertuig', 'totaal_voertuig']
    per_wvk[voertuig_cols] = per_wvk[voertuig_cols].fillna(0).round().astype('Int64')
    per_wvk['wvk_id'] = per_wvk['wvk_id'].astype('Int64')
    return per_wvk


def koppel_inweva(df, inweva_per_wvk):
    n_voor = len(df)
    voertuig_cols = ['klein_voertuig', 'middel_voertuig', 'lang_voertuig', 'totaal_voertuig',
                     '_inweva_omschrijving']
    df = df.drop(columns=[c for c in voertuig_cols if c in df.columns])
    df['wvk_id'] = pd.to_numeric(df['wvk_id'], errors='coerce').astype('Int64')
    df = df.merge(inweva_per_wvk, on='wvk_id', how='left')
    assert len(df) == n_voor, f'Hectopunten omvang VERANDERD! {n_voor} -> {len(df)}'
    return df


download_inweva()
netwerk = lees_netwerk()
werkdag = lees_werkdag()
inweva_per_wvk = bouw_inweva_per_wvk(netwerk, werkdag)

df_voor = len(df)
df = koppel_inweva(df, inweva_per_wvk)
df = df.loc[:, ~df.columns.duplicated()].copy()

# VBNOMSTXT als steekwoord/notitie aan info plakken
df['info'] = df['info'].fillna('').astype(str)
omschr = df['_inweva_omschrijving'].fillna('').astype(str)
df['info'] = df['info'] + omschr.where(omschr == '', ' | ' + omschr).where(df['info'] != '', omschr)
df = df.drop(columns=['_inweva_omschrijving'])

cols = [
    'wvk_id', 'wegnr_hmp', 'Zijde', 'hectomtrng', 'hecto_lttr',
    'klein_voertuig', 'middel_voertuig', 'lang_voertuig',
    'totaal_voertuig', 'distrnaam', 'info', 'streetsmart_link', 'google_maps_link', 'pdok_viewer_link', '|', 'wegbehnaam',
    'beginkm', 'eindkm', 'snelwegnummer', 'wegnummer', 'wegnr_aw', 'rijrichtng', 'oplopend', 'afstand',
    'hectopunt_geometry', 'wegvak_geometry',
    'gps_coordinaten', 'hectopunt_rd_coordinaten',
]
df = df[cols]
df = df.loc[:, ~df.columns.duplicated()].copy()

df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, {df.shape[1]} kolommen')
n_match = df['totaal_voertuig'].notna().sum()
print(f'Hectopunten voor merge : {df_voor:,}')
print(f'Hectopunten na merge   : {df.shape[0]:,}')
print(f'Hectopunten mét INWEVA : {n_match:,} ({n_match/len(df)*100:.1f}%)')
print(f'Kolommen na merge ({len(df.columns)}): {df.columns.tolist()}')

# Bochten

In [ ]:
bochten = gpd.read_file("Input/Bochten/bochten_w.shp")
bochten.rename(columns={'WVK_ID': 'wvk_id'}, inplace=True)
bochten['wvk_id'] = bochten['wvk_id'].astype(float)
print(f'Bochten geladen: {bochten.shape[0]:,} rijen, {bochten.shape[1]} kolommen')

hecto_wvk   = {int(x) for x in df['wvk_id'].dropna().tolist()}
bocht_wvk   = {int(x) for x in bochten['wvk_id'].dropna().tolist()}
overlap_wvk = hecto_wvk & bocht_wvk
print(f'Hectopunten (wvk_ids): {len(hecto_wvk):,}')
print(f'Bochten     (wvk_ids): {len(bocht_wvk):,}')

_hecto_voor   = len(df)
_bochten_voor = len(bochten)

transformer = Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True)


def _coords(geom):
    if geom is None:
        return None
    if geom.geom_type == 'LineString':
        return list(geom.coords)
    if geom.geom_type == 'MultiLineString':
        return [c for line in geom.geoms for c in line.coords]
    return None


def rd_coords_flat(geom):
    coords = _coords(geom)
    if coords is None:
        return None
    return ", ".join(f"{round(x,3)}, {round(y,3)}" for x, y in coords)


def gps_coords_flat(geom):
    coords = _coords(geom)
    if coords is None:
        return None
    return ", ".join(
        f"{round(lat,6)}, {round(lon,6)}"
        for x, y in coords
        for lon, lat in [transformer.transform(x, y)]
    )


df['hectopunt_geometry_flat'] = df['hectopunt_geometry'].apply(
    lambda g: f"{repr(g.geoms[0].x)}, {repr(g.geoms[0].y)}"
)
bochten['bochten_geometry_flat']   = bochten['geometry'].apply(
    lambda g: ', '.join(f"{x}, {y}" for x, y in g.coords)
)
bochten['bochten_rd_coordinaten']  = bochten['geometry'].apply(rd_coords_flat)
bochten['bochten_gps_coordinaten'] = bochten['geometry'].apply(gps_coords_flat)

from scipy.spatial.distance import cdist
from collections import defaultdict


def parse_first_rd_point(geom_str):
    parts = str(geom_str).split(',')
    return float(parts[0].strip()), float(parts[1].strip())


def parse_middle_rd_point(geom_str):
    parts  = [float(p.strip()) for p in str(geom_str).split(',')]
    coords = [(parts[i], parts[i + 1]) for i in range(0, len(parts) - 1, 2)]
    return coords[len(coords) // 2]


df['_rd_x'], df['_rd_y'] = zip(*df['hectopunt_geometry_flat'].apply(parse_first_rd_point))
bochten['_rd_x'], bochten['_rd_y'] = zip(*bochten['bochten_geometry_flat'].apply(parse_middle_rd_point))

df = df.reset_index(drop=True)

# 1) per bocht: dichtstbijzijnde hectopunt binnen wvk_id, max 250 m
hecto_per_wvk = {wvk: sub for wvk, sub in df.groupby('wvk_id')}

bocht_naar_hecto = {}
for bocht_idx, brow in bochten.iterrows():
    sub = hecto_per_wvk.get(brow['wvk_id'])
    if sub is None or sub.empty:
        continue
    bpt       = np.array([[brow['_rd_x'], brow['_rd_y']]])
    hpts      = sub[['_rd_x', '_rd_y']].values
    distances = cdist(bpt, hpts)[0]
    nearest   = np.argmin(distances)
    if distances[nearest] <= 250:
        bocht_naar_hecto[bocht_idx] = sub.index[nearest]

# 2) per hectopunt: alle bochten verzamelen
hecto_naar_bochten = defaultdict(list)
for bocht_idx, hecto_pos in bocht_naar_hecto.items():
    hecto_naar_bochten[hecto_pos].append(bocht_idx)

# 3) per hectopunt: comma-strings opbouwen
def _join(idxs, kol):
    return ', '.join(str(bochten.loc[i, kol]) for i in idxs)


draaihoek_kol, boogstraal_kol     = [], []
geom_flat_kol, rd_kol, gps_kol    = [], [], []
check_lijn_kol, aantal_kol        = [], []

for pos in range(len(df)):
    idxs = hecto_naar_bochten.get(pos, [])
    aantal_kol.append(len(idxs))
    if not idxs:
        draaihoek_kol.append(None);  boogstraal_kol.append(None)
        geom_flat_kol.append(None);  rd_kol.append(None)
        gps_kol.append(None);        check_lijn_kol.append(None)
        continue
    draaihoek_kol .append(_join(idxs, 'DRAAIHOEK'))
    boogstraal_kol.append(_join(idxs, 'BOOGSTRAAL'))
    geom_flat_kol .append(_join(idxs, 'bochten_geometry_flat'))
    rd_kol        .append(_join(idxs, 'bochten_rd_coordinaten'))
    gps_kol       .append(_join(idxs, 'bochten_gps_coordinaten'))
    hx, hy = df.loc[pos, '_rd_x'], df.loc[pos, '_rd_y']
    check_lijn_kol.append(', '.join(
        f"LINESTRING ({hx} {hy}, {bochten.loc[i, '_rd_x']} {bochten.loc[i, '_rd_y']})"
        for i in idxs
    ))

df['draaihoek']               = draaihoek_kol
df['boogstraal']              = boogstraal_kol
df['bochten_geometry_flat']   = geom_flat_kol
df['bochten_rd_coordinaten']  = rd_kol
df['bochten_gps_coordinaten'] = gps_kol
df['aantal_bochten']          = aantal_kol
df['check_koppeling_lijn']    = check_lijn_kol

df.drop(columns=['_rd_x', '_rd_y'], inplace=True)
def _som_getallen(waarde):
    if pd.isna(waarde):
        return None
    return sum(float(x.strip()) for x in str(waarde).split(',') if x.strip())

df['som_draaihoek']  = df['draaihoek'].apply(_som_getallen)
df['som_boogstraal'] = df['boogstraal'].apply(_som_getallen)
bochten.drop(columns=['_rd_x', '_rd_y'], inplace=True)
df = df.sort_values('aantal_bochten', ascending=False).reset_index(drop=True)
df = df[['wvk_id', 'wegnr_hmp', 'Zijde', 'hectomtrng', 'hecto_lttr',
         'draaihoek', 'som_draaihoek',
         'boogstraal', 'som_boogstraal',
         'klein_voertuig', 'middel_voertuig', 'lang_voertuig', 'totaal_voertuig',
         'distrnaam', 'info', 'streetsmart_link', 'google_maps_link', 'pdok_viewer_link', '|',
         'wegbehnaam', 'beginkm', 'eindkm', 'snelwegnummer', 'wegnummer',
         'wegnr_aw', 'rijrichtng', 'oplopend', 'afstand',
         'hectopunt_geometry', 'wegvak_geometry', 'gps_coordinaten',
         'hectopunt_rd_coordinaten', 'hectopunt_geometry_flat',
         'bochten_geometry_flat', 'bochten_rd_coordinaten', 'bochten_gps_coordinaten',
         'aantal_bochten', 'check_koppeling_lijn']]
df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, \
{df.shape[1]} kolommen')

# 4) diagnostiek
hecto_match   = sum(1 for v in aantal_kol if v > 0)
hecto_geen    = len(df) - hecto_match
bochten_match = len(bocht_naar_hecto)
bochten_geen  = _bochten_voor - bochten_match
max_per_hecto = max(aantal_kol) if aantal_kol else 0

print(f'── Hectopunten ──────────────────────────────')
print(f'  Voor koppeling   : {_hecto_voor:,}')
print(f'  Na koppeling     : {len(df):,}')
print(f'  Met bocht(en)    : {hecto_match:,}  ({hecto_match/len(df)*100:.1f}%)')
print(f'  Zonder bocht     : {hecto_geen:,}  ({hecto_geen/len(df)*100:.1f}%)')
print(f'  Max bochten/hecto: {max_per_hecto}')
print(f'── Bochten ──────────────────────────────────')
print(f'  Voor koppeling   : {_bochten_voor:,}')
print(f'  Gekoppeld        : {bochten_match:,}  ({bochten_match/_bochten_voor*100:.1f}%)')
print(f'  Ongekoppeld      : {bochten_geen:,}  ({bochten_geen/_bochten_voor*100:.1f}%)')
print(f'Kolommen na koppeling ({len(df.columns)}): {df.columns.tolist()}')
# display(df.head(1).T)

# Inspectigence

In [ ]:
PAD_2023 = "Input/Wegmarkeringen/Inspectigence-2023/csv/Inspectigence_Part04_2023.csv"
PAD_2025 = "Input/Wegmarkeringen/Inspectigence-2025/csv/Inspectigence_Part04_2025.csv"

INSPECTIES_PUNTEN_PAD       = "Output/qgis_inspecties_punten.gpkg"
INSPECTIES_VERBINDINGEN_PAD = "Output/qgis_inspecties_verbindingen.gpkg"
HECTOPUNTEN_PAD             = "Output/qgis_hectopunten_punten.gpkg"

from shapely.geometry import Point

transformer_to_rd = Transformer.from_crs('EPSG:4326', 'EPSG:28992', always_xy=True)


def lees_inspectigence(pad):
    ins = pd.read_csv(pad)
    ins = ins.rename(columns={'Health_Score': 'health', 'Visibility_Score': 'visibility'})
    ins = ins.drop_duplicates(subset=['KeyID'], keep='first').reset_index(drop=True)
    return ins[['KeyID', 'Geometry', 'health', 'visibility', 'BENAMING']]


def parse_geom(g):
    return g if isinstance(g, BaseGeometry) else wkt.loads(g)


def naar_rd(geom):
    # Inspectigence-Geometry is EPSG:4326 (lon/lat). Reproject naar EPSG:28992 (RD),
    # zodat alle geometrieen in dezelfde CRS staan als hectopunt_geometry.
    return shapely_transform(transformer_to_rd.transform, geom)


def linestring_midpunt(geom_rd):
    # Echt midpunt langs de lijn (geen vertex). Bij 2-punts LineStrings was coords[1]
    # het eindpunt; interpolate(0.5) lost dat op. Input al in RD.
    return geom_rd.interpolate(0.5, normalized=True)


def parse_hectopunt_rd(geom):
    pt = geom.geoms[0] if hasattr(geom, 'geoms') else geom
    return pt.x, pt.y


def sorteer_csv(s, ascending=True):
    if pd.isna(s):
        return s
    vals = sorted((float(v.strip()) for v in str(s).split(',') if v.strip()),
                  reverse=not ascending)
    return ', '.join(f'{v}' for v in vals)


def eerste_waarde(s, default):
    return float(str(s).split(',')[0]) if pd.notna(s) else default


def koppel_inspectigence(df, ins, jaar, hecto_xy, tree):
    """Nearest-match inspecties -> hectopunten. Geeft df met scores per hectopunt
       en een QC-tabel terug met 1 rij per inspectie (punt + verbindingslijn)."""
    ins = ins.copy()
    ins['Geometry'] = ins['Geometry'].apply(parse_geom).apply(naar_rd)

    midpunten = ins['Geometry'].apply(linestring_midpunt).tolist()
    ins_xy    = np.array([(p.x, p.y) for p in midpunten])

    _, indices = tree.query(ins_xy, k=1)
    matched_hecto_xy = hecto_xy[indices]

    qc = pd.DataFrame({
        'jaar':        jaar,
        'KeyID':       ins['KeyID'].values,
        'health':      ins['health'].values,
        'visibility':  ins['visibility'].values,
        'BENAMING':    ins['BENAMING'].values,
        '_hecto_idx':  indices,
        'punt':        midpunten,
        'verbinding': [LineString([(ix, iy), (hx, hy)])
                       for (ix, iy), (hx, hy) in zip(ins_xy, matched_hecto_xy)],
    })

    join_asc      = lambda s: ', '.join(f'{v}' for v in sorted(float(x) for x in s.dropna().tolist()))
    join_benaming = lambda s: ' | '.join(sorted(set(s.dropna().astype(str).tolist())))

    grouped = (
        qc
        .groupby('_hecto_idx', as_index=False)
        .agg({'health':     join_asc,
              'visibility': join_asc,
              'BENAMING':   join_benaming})
    )
    grouped[f'_n_{jaar}'] = qc.groupby('_hecto_idx').size().values
    grouped = grouped.rename(columns={
        'health':     f'health_{jaar}',
        'visibility': f'visibility_{jaar}',
        'BENAMING':   f'_benaming_{jaar}',
    })

    df = df.reset_index(drop=True)
    df['_hecto_idx'] = df.index
    df = df.merge(grouped, on='_hecto_idx', how='left').drop(columns=['_hecto_idx'])
    return df, qc


def voeg_benaming_aan_info(df):
    df['info'] = df['info'].fillna('').astype(str)
    samen = []
    for ben23, ben25 in zip(df['_benaming_2023'].fillna(''), df['_benaming_2025'].fillna('')):
        delen = [b for b in (ben23, ben25) if b]
        samen.append(' | '.join(sorted(set(' | '.join(delen).split(' | ')))) if delen else '')
    extra = pd.Series(samen, index=df.index)
    df['info'] = np.where(
        extra == '', df['info'],
        np.where(df['info'] == '', extra, df['info'] + ' | ' + extra)
    )
    return df.drop(columns=['_benaming_2023', '_benaming_2025'])


def diagnose(df, ins, jaar):
    n_h     = len(df)
    n_match = df[f'health_{jaar}'].notna().sum()
    print(f'-- Inspectigence {jaar} ------------------------')
    print(f'  Hectopunten              : {n_h:,}')
    print(f'  Inspectigence-records    : {len(ins):,}')
    print(f'  Hectopunten met match    : {n_match:,} ({n_match/n_h*100:.1f}%)')
    print(f'  Hectopunten zonder match : {n_h - n_match:,} ({(n_h - n_match)/n_h*100:.1f}%)')


# 0) Schone start -- ook oude geo-kolommen weg, die staan nu in losse GPKG-lagen
oud = [c for c in df.columns
       if c.startswith(('ins_', 'ins23_', 'ins25_', 'ins2023_', 'ins2025_'))
       or c in ('health_2023', 'visibility_2023', 'health_2025', 'visibility_2025',
                'aantal_inspecties', '_benaming_2023', '_benaming_2025',
                'geometry_2023', 'geometry_2025',
                'verbindingslijn_2023', 'verbindingslijn_2025')]
df = df.drop(columns=oud, errors='ignore')

# 1) KDTree over hectopunten -- eenmalig, hergebruikt voor beide jaren
hecto_xy = np.array(df['hectopunt_geometry'].apply(parse_hectopunt_rd).tolist())
tree     = cKDTree(hecto_xy)

# 1b) Schrijf de hectopunten-puntenlaag uit EXACT dezelfde hecto_xy als de
#     KDTree. Garantie: elk verbindingslijn-eindpunt valt op een hectopunt
#     in deze laag -- nooit meer 'lijnen naar het niets' in QGIS.
_hecto_attr_cols = [c for c in ['wegnr_hmp', 'hectomtrng', 'hecto_lttr',
                                'Zijde', 'wvk_id', 'distrnaam', 'deklaagsoort']
                    if c in df.columns]
hectopunten_punten_laag = gpd.GeoDataFrame(
    {**{c: df[c].values for c in _hecto_attr_cols},
     '_hecto_idx': np.arange(len(df))},
    geometry=[Point(x, y) for x, y in hecto_xy],
    crs='EPSG:28992',
)
hectopunten_punten_laag.to_file(HECTOPUNTEN_PAD, driver='GPKG')
print(f'Hectopunten-puntenlaag : {HECTOPUNTEN_PAD}  '
      f'({len(hectopunten_punten_laag):,} punten)')

# 2) Inlezen + koppelen; QC-records (punt + verbindingslijn per inspectie) opzij houden
_hecto_voor = len(df)
ins_2023 = lees_inspectigence(PAD_2023)
ins_2025 = lees_inspectigence(PAD_2025)
print(f'Inspectigence 2023 : {len(ins_2023):,} records')
print(f'Inspectigence 2025 : {len(ins_2025):,} records')

df, qc_2023 = koppel_inspectigence(df, ins_2023, jaar=2023, hecto_xy=hecto_xy, tree=tree)
df, qc_2025 = koppel_inspectigence(df, ins_2025, jaar=2025, hecto_xy=hecto_xy, tree=tree)
assert len(df) == _hecto_voor, f'Hectopunten omvang VERANDERD! {_hecto_voor} -> {len(df)}'

# 3) Testcase-kaartlagen wegschrijven (EPSG:28992, gelijk aan hectopunt_geometry):
#    - puntenlaag : 1 rij per inspectie (geom + jaar + health + visibility + BENAMING)
#    - lijnenlaag : 1 rij per koppeling (inspectie-midpunt -> gekoppelde hectopunt)
#    Zo zie je in QGIS direct welke hectopunt aan welke inspectie hangt.
qc_alle = pd.concat([qc_2023, qc_2025], ignore_index=True)

inspecties_punten = gpd.GeoDataFrame(
    qc_alle[['jaar', 'KeyID', 'health', 'visibility', 'BENAMING', '_hecto_idx']],
    geometry=qc_alle['punt'].tolist(),
    crs='EPSG:28992',
)
inspecties_verbindingen = gpd.GeoDataFrame(
    qc_alle[['jaar', 'KeyID', '_hecto_idx']],
    geometry=qc_alle['verbinding'].tolist(),
    crs='EPSG:28992',
)
inspecties_punten.to_file(INSPECTIES_PUNTEN_PAD, driver='GPKG')
inspecties_verbindingen.to_file(INSPECTIES_VERBINDINGEN_PAD, driver='GPKG')
print(f'QC-puntenlaag       : {INSPECTIES_PUNTEN_PAD}  ({len(inspecties_punten):,} punten)')
print(f'QC-verbindingenlaag : {INSPECTIES_VERBINDINGEN_PAD}  ({len(inspecties_verbindingen):,} lijnen)')

# 4) BENAMING aan info plakken (uniek, ' | '-gescheiden)
df = voeg_benaming_aan_info(df)

# 5) aantal_inspecties = max over beide jaren
df['aantal_inspecties'] = df[['_n_2023', '_n_2025']].max(axis=1).astype('Int64')
df = df.drop(columns=['_n_2023', '_n_2025'])

# 6) Per cel intern sorteren zodat [0] altijd het juiste uiterste is
df['draaihoek']  = df['draaihoek'].apply(lambda s: sorteer_csv(s, ascending=False))
df['boogstraal'] = df['boogstraal'].apply(lambda s: sorteer_csv(s, ascending=True))

# 7) Kolomvolgorde: scores na boogstraal, aantal_inspecties achteraan
score_cols = ['health_2023', 'visibility_2023', 'health_2025', 'visibility_2025']
for i, c in enumerate(score_cols):
    df.insert(df.columns.get_loc('boogstraal') + 1 + i, c, df.pop(c))
df['aantal_inspecties'] = df.pop('aantal_inspecties')

# 8) Sorteren df: hoogste hoek boven, dan laagste visibility_2025
df = (df
      .assign(
          _max_hoek=df['draaihoek'].apply(lambda s: eerste_waarde(s, -np.inf)),
          _min_vis =df['visibility_2025'].apply(lambda s: eerste_waarde(s,  np.inf)),
      )
      .sort_values(['_max_hoek', '_min_vis'], ascending=[False, True])
      .drop(columns=['_max_hoek', '_min_vis'])
      .reset_index(drop=True))

df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, {df.shape[1]} kolommen')

# 9) Diagnose
print()
diagnose(df, ins_2023, 2023)
print()
diagnose(df, ins_2025, 2025)
print()
print(f'Hectopunten voor koppeling : {_hecto_voor:,}')
print(f'Hectopunten na koppeling   : {len(df):,}')
print(f'Max inspecties per hecto   : {df["aantal_inspecties"].max()}')
print(f'Kolommen na koppeling ({len(df.columns)}): {df.columns.tolist()}')


In [ ]:
# Testcase: laad de twee QC-kaartlagen terug en toon stats + sample.
# Beide lagen in EPSG:28992. In QGIS open je de puntenlaag samen met de
# verbindingenlaag en hectopunten om te zien of de nearest-koppeling klopt.
qc_punten       = gpd.read_file(INSPECTIES_PUNTEN_PAD)
qc_verbindingen = gpd.read_file(INSPECTIES_VERBINDINGEN_PAD)
print(f'{INSPECTIES_PUNTEN_PAD:50s} -> {len(qc_punten):,} punten,  CRS {qc_punten.crs.to_string()}')
print(f'{INSPECTIES_VERBINDINGEN_PAD:50s} -> {len(qc_verbindingen):,} lijnen,  CRS {qc_verbindingen.crs.to_string()}')
assert len(qc_punten) == len(qc_verbindingen), 'Punt- en verbindingen-laag moeten 1-op-1 zijn'
display(qc_punten.sample(min(3, len(qc_punten))))
display(qc_verbindingen.sample(min(3, len(qc_verbindingen))))


# Deklagen

In [ ]:
from pathlib import Path
import pandas as pd

DEKLAGEN_XLS = Path('Input/Deklagen/Deklagen_M26_NL_Totaal Definitief.xls')


def lees_deklagen():
    with Duur('Inlezen Deklagenlijst'):
        dek = pd.read_excel(DEKLAGEN_XLS, sheet_name='DeklagenM26Totaal')
    dek = dek.dropna(subset=['VAN', 'TOT']).copy()
    dek['AANLEGDATUM'] = pd.to_datetime(dek['AANLEGDATUM'], format='%d-%m-%Y', errors='coerce')
    dek['wegnr_hmp'] = dek['WEG'].astype(int).apply(lambda n: f'A{n}')
    # VAN/TOT zijn in km; hectomtrng is in hectometers (×10). Conversie voor range-join.
    dek['VAN_hm'] = (dek['VAN'] * 10).round().astype(int)
    dek['TOT_hm'] = (dek['TOT'] * 10).round().astype(int)
    return dek


def koppel_deklagen(df, dek):
    """
    Range-join: koppel elke hectopunt aan deklagen waarvan het km-bereik dat punt dekt.
    Voorwaarde: VAN_hm ≤ hectomtrng ≤ TOT_hm  (beiden in hectometers)
    """
    # Stap 1: kandidaten beperken tot zelfde weg (vermijdt explosief kruis-product)
    kandidaten = df[['wvk_id', 'wegnr_hmp', 'hectomtrng']].merge(
        dek, on='wegnr_hmp', how='inner'
    )
    # Stap 2: range-filter
    masker = (
        (kandidaten['hectomtrng'] >= kandidaten['VAN_hm']) &
        (kandidaten['hectomtrng'] <= kandidaten['TOT_hm'])
    )
    matches = kandidaten[masker].drop_duplicates(
        subset=['wvk_id', 'hectomtrng', 'BAAN', 'STROOK', 'VAN', 'TOT', 'AANLEGDATUM']
    )
    return matches.sort_values('AANLEGDATUM')


def slimme_join(s):
    vals = s.dropna().astype(str).tolist()
    if not vals:
        return None
    unieke = list(dict.fromkeys(vals))
    if len(unieke) == 1:
        return unieke[0]
    if all(v == 'ALL' for v in unieke):
        return 'ALL'
    return ', '.join(unieke)


def aggregeer_per_hectopunt(matches):
    return (
        matches
        .groupby(['wvk_id', 'hectomtrng'], as_index=False)
        .agg(
            deklaagsoort    = ('DEKLAAGSOORT', slimme_join),
            aanlegdatum     = ('AANLEGDATUM', lambda s: ', '.join(
                s.dt.strftime('%Y-%m-%d').dropna().unique()
            )),
            strook          = ('STROOK', slimme_join),
            aantal_deklagen = ('DEKLAAGSOORT', 'size'),
        )
    )


def diagnose(df, dek, matches):
    n_h      = len(df)
    n_match  = df['deklaagsoort'].notna().sum()
    wegen_in_dek = set(dek['wegnr_hmp'])
    wegen_in_df  = set(df['wegnr_hmp'].dropna())
    ontbreekt    = wegen_in_df - wegen_in_dek

    print(f'── Deklagen ─────────────────────────────────')
    print(f'  Hectopunten              : {n_h:,}')
    print(f'  Deklagen-records         : {len(dek):,}')
    print(f'  Range-matches (lang)     : {len(matches):,}')
    print(f'  Hectopunten met deklaag  : {n_match:,} ({n_match/n_h*100:.1f}%)')
    print(f'  Hectopunten zonder       : {n_h - n_match:,} ({(n_h - n_match)/n_h*100:.1f}%)')
    print(f'  Max deklagen / hecto     : {df["aantal_deklagen"].max()}')
    print(f'  Mediaan deklagen / hecto : {df["aantal_deklagen"].median()}')
    print(f'  Wegen in df              : {sorted(wegen_in_df)}')
    print(f'  Wegen in df NIET in dek  : {sorted(ontbreekt) if ontbreekt else "geen"}')


# ─────────────────────────────────────────────
# PIPELINE
# ─────────────────────────────────────────────

# 0) Schone start
df = df.drop(columns=['deklaagsoort', 'aanlegdatum', 'strook', 'aantal_deklagen'],
             errors='ignore')

# 1) Inlezen (VAN_hm / TOT_hm worden aangemaakt in lees_deklagen)
_hecto_voor = len(df)
dek = lees_deklagen()
print(f'Deklagen-records         : {len(dek):,}')

# 2) Koppelen via range-join + aggregeren per hectopunt
matches     = koppel_deklagen(df, dek)
deklagen_hp = aggregeer_per_hectopunt(matches)
print(f'Range-matches            : {len(matches):,}')
print(f'Unieke hectopunten match : {len(deklagen_hp):,}')

# 3) Merge terug op hectopunten (geen toename in rijen)
df = df.merge(deklagen_hp, on=['wvk_id', 'hectomtrng'], how='left')
assert len(df) == _hecto_voor, f'Hectopunten omvang VERANDERD! {_hecto_voor} -> {len(df)}'

# 4) Kolomvolgorde
nieuwe_cols = ['deklaagsoort', 'aanlegdatum', 'strook']
for i, c in enumerate(nieuwe_cols):
    df.insert(df.columns.get_loc('visibility_2025') + 1 + i, c, df.pop(c))
df['aantal_deklagen'] = df.pop('aantal_deklagen')

# 5) Info opschonen
df['info'] = df['info'].fillna('').astype(str).apply(
    lambda s: ' | '.join(dict.fromkeys(p.strip() for p in s.split('|') if p.strip()))
)

# 6) Opslaan + laden
df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, {df.shape[1]} kolommen')

# 7) Diagnose
print()
diagnose(df, dek, matches)
print()
print(f'Hectopunten voor koppeling : {_hecto_voor:,}')
print(f'Hectopunten na koppeling   : {len(df):,}')
print(f'Kolommen na koppeling ({len(df.columns)}): {df.columns.tolist()}')


# Levensduur

In [ ]:
# Verrijken met levensduur op basis van draaihoek (bocht) + zwaar verkeer (lang_voertuig)
# Waarheidstabel:
#   scherpe bocht (>=30°) én zwaar > x   ->  4 jaar
#   scherpe bocht (>=30°) én zwaar <= x  ->  6 jaar
#   geen scherpe bocht    én zwaar > x   ->  8 jaar
#   geen scherpe bocht    én zwaar <= x  -> 12 jaar
# scherpe bocht = bocht aanwezig én som draaihoek >= 30°
# x = drempel top 10% = 0.9 * hoogste zwaar verkeer (lang_voertuig)

def _som_draaihoek(s):
    # draaihoek is een comma-string met 0..n hoeken; tel alle hoeken in de rij op
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return np.nan
    waarden = [float(p.strip()) for p in str(s).split(',') if p.strip() != '']
    return sum(waarden) if waarden else np.nan

hoek  = df['draaihoek'].apply(_som_draaihoek)
zwaar = pd.to_numeric(df['lang_voertuig'], errors='coerce')

x = 0.9 * zwaar.max()
print(f'Drempel x (0.9 * max lang_voertuig) = {x:,.1f} voertuigen')

heeft_bocht = hoek.notna().to_numpy(dtype=bool)
hoek_np     = hoek.to_numpy(dtype=float)
veel_zwaar  = (zwaar > x).to_numpy(dtype=bool, na_value=False)

scherp = heeft_bocht & (hoek_np >= 30)   # geen bocht / <30° -> niet scherp

levensduur = np.select(
    [
        scherp  & veel_zwaar,   #  4 jaar
        scherp  & ~veel_zwaar,  #  6 jaar
        ~scherp & veel_zwaar,   #  8 jaar
        ~scherp & ~veel_zwaar,  # 12 jaar
    ],
    [4, 6, 8, 12],
    default=12,
)
df['levensduur'] = pd.array(levensduur, dtype='Int64')

print(df['levensduur'].value_counts(dropna=False))
df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df[['wvk_id', 'wegnr_hmp', 'hectomtrng', '|', 'draaihoek', 'lang_voertuig', 'levensduur']].head(10)

In [ ]:
# Onderhoudsmoment: eerstvolgend onderhoudsjaar >= 2026.
# Start = jaar van nieuwste aanlegdatum deklaag. Tel levensduur (interval) er
# steeds bij op tot het jaar 2026 of later is. Levensduur = onderhoudsinterval.
DOELJAAR = 2026

def _nieuwste_jaar(s):
    # nieuwste aanlegdatum uit de comma-string; geef het jaar terug
    if pd.isna(s):
        return np.nan
    ds = pd.to_datetime([p.strip() for p in str(s).split(',') if p.strip()],
                        errors='coerce')
    ds = ds.dropna()
    return ds.max().year if len(ds) else np.nan

_start_jaar = df['aanlegdatum'].apply(_nieuwste_jaar)
_lev        = pd.to_numeric(df['levensduur'], errors='coerce')

def _onderhoudsmoment(jaar, lev):
    if pd.isna(jaar) or pd.isna(lev) or lev <= 0:
        return pd.NA
    moment = int(jaar)
    while moment < DOELJAAR:
        moment += int(lev)
    return moment

df['onderhoudsmoment'] = pd.array(
    [_onderhoudsmoment(j, l) for j, l in zip(_start_jaar, _lev)],
    dtype='Int64',
)

print(df['onderhoudsmoment'].value_counts(dropna=False).sort_index())

df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df[['snelwegnummer', 'Zijde', 'hectomtrng', 'hecto_lttr',
    'som_draaihoek', 'lang_voertuig', 'aanlegdatum', 'onderhoudsmoment']].head(10)

# onderhoudsmoment

In [ ]:
len(df.columns)

# Voorsorteren (leesbaarheid)

Eindsortering vóór opslaan. Eerst netwerk-volgorde, daarna binnen elke groep op draaihoek, lang_voertuig en nieuwste aanlegdatum deklaag (hoogste/nieuwste boven, NaN's onderaan).

In [ ]:
# Voorsorteren voor leesbaarheid in de tabel.
# Sorteervolgorde:
# 1) tier  -> rijen met meeste info bovenaan:
#       0 = onderhoudsmoment EN hoek bekend
#       1 = onderhoudsmoment NaN, hoek bekend
#       2 = hoek NaN
# 2) onderhoudsmoment (oplopend)
# 3) hoek-bin (aflopend): som |hoek| > 30 bovenop som |hoek| <= 30
# 4) totaal_voertuig (aflopend): binnen elke bin meeste verkeer bovenop
# 5) som_draaihoek (aflopend): grootste totale |hoek| als laatste tiebreak
# Bins/temp-kolommen alleen voor sortering, niet in output.

def _som_abs_hoek(s):
    if pd.isna(s):
        return np.nan
    vals = [abs(float(p.strip())) for p in str(s).split(',') if p.strip()]
    return sum(vals) if vals else np.nan

_temp_cols = ['_hoek_sort', '_hoek_bin', '_tier']

_hoek_sort = df['som_draaihoek'].apply(_som_abs_hoek)
_moment_na = df['onderhoudsmoment'].isna()
_hoek_na   = _hoek_sort.isna()

# tier: 0 = beide bekend, 1 = moment NaN (hoek bekend), 2 = hoek NaN
_tier = np.where(_hoek_na, 2, np.where(_moment_na, 1, 0))

# hoek-bin: 1 = scherpe hoek (> 30), 0 = flauw (<= 30); NaN blijft NaN
_hoek_bin = np.where(_hoek_na, np.nan, (_hoek_sort > 30).astype(float))

df = (df
      .assign(
          _hoek_sort = _hoek_sort,
          _hoek_bin  = _hoek_bin,
          _tier      = _tier,
      )
      .sort_values(
          ['_tier', 'onderhoudsmoment', '_hoek_bin', 'totaal_voertuig', '_hoek_sort'],
          ascending=[True, True, False, False, False],
          na_position='last',
          kind='mergesort',
      )
      .drop(columns=_temp_cols)
      .reset_index(drop=True))

df.to_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Voorgesorteerd & opgeslagen: {len(df):,} rijen')
df[['snelwegnummer', 'Zijde', 'hectomtrng', 'hecto_lttr',
    'som_draaihoek', 'lang_voertuig', 'aanlegdatum', 'onderhoudsmoment']].head(10)

In [ ]:
df.columns

In [ ]:
# df.head(1).T

In [ ]:
# doorzetten naar dashboard.